# 5. Reopen, inspect, and clean the downloaded data

A new notebook starts a new Python session. Variables from the previous notebook do **not** carry over.

Instead, we reopen the file saved in the previous step.

In [ ]:
from pathlib import Path

import geopandas as gpd

In [ ]:
raw_osm_file = Path("../data/raw/toronto_osm_libraries.gpkg")
practice_file = Path("../data/practice/toronto_library_points_practice.geojson")

if raw_osm_file.exists():
    libraries = gpd.read_file(raw_osm_file)
    print("Using saved live OSM data.")
else:
    libraries = gpd.read_file(practice_file)
    print("Live OSM file not found — using the course practice dataset.")

## Inspect the structure

Ask the same questions we used throughout the course:

- How many rows and columns?
- What are the data types?
- Which columns are useful?
- Which values are missing?
- What geometry types are present?

In [ ]:
print("Shape:", libraries.shape)
print("CRS:", libraries.crs)

print("\nGeometry types:")
print(libraries.geometry.geom_type.value_counts())

print("\nColumns:")
print(libraries.columns.tolist())

In [ ]:
display(libraries.head())

## Why OSM tables can look unusually wide

Different OSM features can have different tags.

One library might have:

```text
name
operator
website
opening_hours
address
```

while another might have only:

```text
name
amenity
```

When many records are combined into one table, this produces many columns containing missing values.

In [ ]:
if "name" in libraries.columns:
    print("Missing names:", libraries["name"].isna().sum())

## Keep only what the analysis needs

We do not need every OSM tag for a basic location analysis.

When working with a live OSMnx result, first reset the index so the OSM identifiers become regular columns.

In [ ]:
libraries_clean = libraries.reset_index()

desired_columns = [
    "element_type",
    "osmid",
    "name",
    "address",
    "geometry"
]

columns_to_keep = []

for column in desired_columns:
    if column in libraries_clean.columns:
        columns_to_keep.append(column)

libraries_clean = libraries_clean[
    columns_to_keep
].copy()

display(libraries_clean.head())

## Save the cleaned file

This creates another explicit checkpoint in the workflow:

```text
live request
    ↓
raw saved data
    ↓
cleaned saved data
    ↓
analysis
```

In [ ]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

clean_file = processed_dir / "toronto_libraries_clean.gpkg"

libraries_clean.to_file(
    clean_file,
    driver="GPKG"
)

print("Saved:", clean_file)